In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import re
import openslide
from os.path import join as opj

In [ ]:
def sanitize_string(string):
    return re.sub(r'[^a-zA-Z0-9]', '', string)

# Visualize image

In [ ]:
all_meta = pd.read_csv('/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/master_spreadsheet/matched_su_mxa.csv')

In [ ]:
currdf = all_meta.loc[all_meta["UM Accession"]=="SU-24-38184"]
#currdf = currdf.loc[currdf["Block"]=="E1"]
#currdf = currdf.loc[currdf["Block"].str.contains("E1")]
#currdf = currdf.loc[currdf["Stain"]=="H&E"]
#currdf = all_meta.loc[all_meta["a_num"].isin(range(11917,11925))]
#currdf = all_meta.loc[all_meta["Barcode"]=="S1230CLXM"]
#currdf = all_meta[(all_meta["m_num"]==-1) & (all_meta["x_num"]==-1)].groupby(["a_num"]).filter(lambda x: len(x) > 1)
#currdf = all_meta.loc[all_meta["a_num"]==8759]
#currdf = all_meta.loc[all_meta["a_num"].isin({18540,18541,18632,18634,18635,18659})]

#currdf = all_meta[all_meta["qr_meta_fn"]=="20250209.csv"]
#currdf = currdf.sort_values(["Block", "Stain"])

#currdf = all_meta[all_meta["Stain"]=="Olig2"].sample(10)
currdf = currdf.sort_values(["Barcode"])
currdf

In [ ]:
svs_root = "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/svs"

for i, s in tqdm(currdf.fillna("").iterrows(), total=len(currdf)):
    if s["path"] == "":
        continue
        
    fig, axs = plt.subplots(1,3,  figsize=(12,3), width_ratios=[1, 3,3])
    
    try:
        slide = openslide.OpenSlide(opj(svs_root, s["path"]))

        axs[0].imshow(slide.associated_images["label"])
        axs[0].set_title(s["mxa_code"])
        axs[1].imshow(slide.associated_images["macro"])
        axs[2].imshow(slide.associated_images["thumbnail"])
        axs[1].set_title(s["path"].split("/")[-1]+"\n"+s['UM Accession']+" / "+s['Block']+" / "+s["Stain"])
    except:
        axs[0].set_title(s["mxa_code"])
        axs[1].set_title(s["path"].split("/")[-1]+"\n"+s['UM Accession']+" / "+s['Block']+" / "+s["Stain"])
        
    for ax in axs: ax.axis("off") 
    
    subs_str = "_".join([s['UM Accession'], s['Block'], sanitize_string(s["Stain"])]).lower()
    
    #slide.associated_images["macro"].save(f"{subs_str}_macro.png")
    #slide.associated_images["thumbnail"].save(f"{subs_str}_thumbnail.png")    

In [ ]:
pd.set_option('display.max_rows', None)
all_meta["Block"][all_meta["Block"].str.endswith("a")].value_counts()